# Tutorial 11C: Four-Component Model and TFP Decomposition — Solutions

**Module**: Stochastic Frontier Analysis | **Difficulty**: Advanced

This notebook contains complete solutions for all exercises in Tutorial 11C.

In [ ]:
# ============================================================
# Setup
# ============================================================

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
from pathlib import Path

from scipy import stats

from panelbox.datasets import load_dataset
from panelbox.frontier import FourComponentSFA

np.random.seed(42)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

BASE_DIR = Path("..")
FIGURES_DIR = BASE_DIR / "outputs" / "figures" / "03_four_comp"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

---

## Exercise 1: Electricity Sector Analysis (Medium)

Load the `electricity_panel.csv` dataset and:
1. Estimate a four-component model for electricity generators
2. Calculate the variance decomposition and compare with manufacturing results
3. Identify generators with structural vs temporary inefficiency
4. Analyze efficiency patterns by fuel type
5. Recommend specific interventions

In [ ]:
# Step 1: Load the electricity data
elec_data = load_dataset("electricity_panel", category="frontier")

print(f"Dataset shape: {elec_data.shape}")
print(f"Variables: {elec_data.columns.tolist()}")
print(f"Generators: {elec_data['generator_id'].nunique()}")
print(f"Years: {elec_data['year'].nunique()} ({elec_data['year'].min()}-{elec_data['year'].max()})")
print(f"Fuel types: {elec_data['fuel_type'].unique().tolist()}")

display(elec_data.head())
display(elec_data.describe())

In [ ]:
# Step 2: Estimate four-component model
elec_model = FourComponentSFA(
    data=elec_data,
    depvar="log_output",
    exog=["log_labor", "log_capital", "log_fuel"],
    entity="generator_id",
    time="year",
    frontier_type="production",
)

elec_result = elec_model.fit(verbose=True)

In [ ]:
# Display results
elec_result.print_summary()

In [ ]:
# Step 2 (cont): Variance decomposition comparison
sv2 = elec_result.sigma_v**2
su2 = elec_result.sigma_u**2
smu2 = elec_result.sigma_mu**2
seta2 = elec_result.sigma_eta**2
total = sv2 + su2 + smu2 + seta2

print("=" * 60)
print("VARIANCE DECOMPOSITION - ELECTRICITY SECTOR")
print("=" * 60)
print(f"\n  {'Component':<30} {'Variance':>10} {'Share':>10}")
print(f"  {'-' * 50}")
print(f"  {'sigma_v^2 (noise)':<30} {sv2:>10.6f} {100 * sv2 / total:>9.1f}%")
print(f"  {'sigma_u^2 (transient ineff.)':<30} {su2:>10.6f} {100 * su2 / total:>9.1f}%")
print(f"  {'sigma_mu^2 (heterogeneity)':<30} {smu2:>10.6f} {100 * smu2 / total:>9.1f}%")
print(f"  {'sigma_eta^2 (persistent ineff.)':<30} {seta2:>10.6f} {100 * seta2 / total:>9.1f}%")

print(f"\n  Key finding: heterogeneity accounts for {100 * smu2 / total:.1f}% of total variance")
print(
    "  This likely reflects fundamental differences between fuel types (coal vs gas vs renewable)"
)

In [ ]:
# Step 3: Identify generators with structural vs temporary problems
pe_elec = elec_result.persistent_efficiency()
te_elec = elec_result.transient_efficiency()
oe_elec = elec_result.overall_efficiency()

print("=" * 60)
print("EFFICIENCY SUMMARY")
print("=" * 60)
eff_summary = pd.DataFrame(
    {
        "Persistent (PE)": pe_elec["persistent_efficiency"].describe(),
        "Transient (TE)": te_elec["transient_efficiency"].describe(),
        "Overall (OE)": oe_elec["overall_efficiency"].describe(),
    }
)
display(eff_summary)

print("\nGenerators with LOW PERSISTENT efficiency (structural problems):")
low_pe = pe_elec.nsmallest(5, "persistent_efficiency")
display(low_pe)

print("\nGenerators with LOW TRANSIENT efficiency (temporary issues):")
avg_te = te_elec.groupby("entity")["transient_efficiency"].mean().reset_index()
avg_te.columns = ["entity", "avg_transient_efficiency"]
low_te = avg_te.nsmallest(5, "avg_transient_efficiency")
display(low_te)

In [ ]:
# Step 4: Efficiency patterns by fuel type
fuel_info = elec_data.groupby("generator_id")["fuel_type"].first().reset_index()

# Merge persistent efficiency with fuel type
pe_fuel = pe_elec.merge(fuel_info, left_on="entity", right_on="generator_id", how="left")

# Average transient efficiency with fuel type
te_avg_fuel = avg_te.merge(fuel_info, left_on="entity", right_on="generator_id", how="left")

print("=" * 60)
print("EFFICIENCY BY FUEL TYPE")
print("=" * 60)
print("\nPersistent Efficiency by Fuel Type:")
pe_by_fuel = pe_fuel.groupby("fuel_type")["persistent_efficiency"].agg(
    ["mean", "std", "min", "max"]
)
display(pe_by_fuel.round(4))

print("\nTransient Efficiency by Fuel Type:")
te_by_fuel = te_avg_fuel.groupby("fuel_type")["avg_transient_efficiency"].agg(
    ["mean", "std", "min", "max"]
)
display(te_by_fuel.round(4))

In [ ]:
# Visualization: Efficiency by fuel type
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Persistent efficiency by fuel type
sns.boxplot(x="fuel_type", y="persistent_efficiency", data=pe_fuel, ax=axes[0], palette="Set2")
axes[0].set_xlabel("Fuel Type", fontsize=12)
axes[0].set_ylabel("Persistent Efficiency", fontsize=12)
axes[0].set_title("Persistent Efficiency by Fuel Type", fontsize=13, fontweight="bold")

# Transient efficiency by fuel type
sns.boxplot(
    x="fuel_type", y="avg_transient_efficiency", data=te_avg_fuel, ax=axes[1], palette="Set2"
)
axes[1].set_xlabel("Fuel Type", fontsize=12)
axes[1].set_ylabel("Avg Transient Efficiency", fontsize=12)
axes[1].set_title("Transient Efficiency by Fuel Type", fontsize=13, fontweight="bold")

plt.suptitle(
    "Electricity Sector: Efficiency Analysis by Fuel Type", fontsize=15, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ex1_efficiency_by_fuel.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Step 5: Recommendations
print("=" * 60)
print("POLICY RECOMMENDATIONS")
print("=" * 60)

# Identify the worst generators and their problems
pe_fuel_sorted = pe_fuel.sort_values("persistent_efficiency")
worst_persistent = pe_fuel_sorted.head(5)

print("\n1. Generators with STRUCTURAL problems (low persistent efficiency):")
for _, row in worst_persistent.iterrows():
    print(
        f"   Generator {row['entity']} ({row['fuel_type']}): PE = {row['persistent_efficiency']:.3f}"
    )
    print("   -> Recommendation: Organizational restructuring, management upgrade")

te_avg_fuel_sorted = te_avg_fuel.sort_values("avg_transient_efficiency")
worst_transient = te_avg_fuel_sorted.head(5)

print("\n2. Generators with TEMPORARY problems (low transient efficiency):")
for _, row in worst_transient.iterrows():
    print(
        f"   Generator {row['entity']} ({row['fuel_type']}): avg TE = {row['avg_transient_efficiency']:.3f}"
    )
    print("   -> Recommendation: Operational improvements, maintenance optimization")

print("\n3. General recommendations by fuel type:")
for fuel in pe_by_fuel.index:
    pe_mean = pe_by_fuel.loc[fuel, "mean"]
    te_mean = te_by_fuel.loc[fuel, "mean"]
    print(f"   {fuel}: PE={pe_mean:.3f}, TE={te_mean:.3f}")
    if pe_mean < te_mean:
        print("   -> Main issue is structural. Focus on technology upgrades and management.")
    else:
        print("   -> Main issue is operational. Focus on day-to-day improvements.")

---

## Exercise 2: TFP Decomposition of Electricity Sector (Medium)

Using the electricity model:
1. Decompose TFP growth
2. Determine the main source: innovation or catch-up?
3. Analyze differences by fuel type
4. Create scatter plot of ΔTC vs ΔTE colored by fuel type
5. Compare with manufacturing results

In [ ]:
# Step 1: TFP decomposition
elec_years = sorted(elec_data["year"].unique())
print(f"Years: {elec_years[0]} -> {elec_years[-1]}")

elec_tfp = elec_result.tfp_decomposition(periods=(elec_years[0], elec_years[-1]))
elec_decomp = elec_tfp.decompose()

print(f"\nDecomposition shape: {elec_decomp.shape}")
display(elec_decomp.describe())

In [ ]:
# Step 2: Aggregate results and identify main driver
elec_agg = elec_tfp.aggregate_decomposition()

print("=" * 60)
print("AGGREGATE TFP DECOMPOSITION - ELECTRICITY")
print("=" * 60)
print(f"\n  Mean TFP Growth:     {elec_agg['mean_delta_tfp']:.4f}")
print(f"  Technical Change:    {elec_agg['mean_delta_tc']:.4f}  ({elec_agg['pct_from_tc']:.1f}%)")
print(f"  Efficiency Change:   {elec_agg['mean_delta_te']:.4f}  ({elec_agg['pct_from_te']:.1f}%)")
print(f"  Scale Effect:        {elec_agg['mean_delta_se']:.4f}  ({elec_agg['pct_from_se']:.1f}%)")

# Determine main driver
drivers = {
    "Innovation (TC)": abs(elec_agg["mean_delta_tc"]),
    "Catch-up (TE)": abs(elec_agg["mean_delta_te"]),
    "Scale (SE)": abs(elec_agg["mean_delta_se"]),
}
main_driver = max(drivers, key=drivers.get)
print(f"\n  Main driver of TFP growth: {main_driver}")

In [ ]:
# Step 3: Analysis by fuel type
elec_decomp_merged = elec_decomp.merge(
    fuel_info, left_on="entity", right_on="generator_id", how="left"
)

print("=" * 60)
print("TFP DECOMPOSITION BY FUEL TYPE")
print("=" * 60)
fuel_decomp = elec_decomp_merged.groupby("fuel_type")[
    ["delta_tfp", "delta_tc", "delta_te", "delta_se"]
].mean()
display(fuel_decomp.round(4))

print("\nInterpretation:")
for fuel in fuel_decomp.index:
    row = fuel_decomp.loc[fuel]
    print(f"\n  {fuel}:")
    print(f"    TFP growth: {row['delta_tfp']:.4f}")
    if abs(row["delta_tc"]) > abs(row["delta_te"]):
        print(f"    -> Innovation-driven (ΔTC = {row['delta_tc']:.4f})")
    else:
        print(f"    -> Catch-up driven (ΔTE = {row['delta_te']:.4f})")

In [ ]:
# Step 4: Scatter plot ΔTC vs ΔTE colored by fuel type
fig, ax = plt.subplots(figsize=(10, 8))

fuel_colors = {"coal": "#8B4513", "gas": "#3498db", "hydro": "#2ecc71", "nuclear": "#f39c12"}

for fuel_type in elec_decomp_merged["fuel_type"].unique():
    subset = elec_decomp_merged[elec_decomp_merged["fuel_type"] == fuel_type]
    color = fuel_colors.get(fuel_type, "gray")
    ax.scatter(
        subset["delta_tc"],
        subset["delta_te"],
        label=f"{fuel_type} (n={len(subset)})",
        alpha=0.6,
        s=80,
        color=color,
        edgecolors="black",
        linewidth=0.3,
    )

ax.axhline(0, color="black", linewidth=0.8, alpha=0.3)
ax.axvline(0, color="black", linewidth=0.8, alpha=0.3)

# Quadrant labels
ax.text(
    0.02,
    0.98,
    "Innovation\n+ Catch-up",
    transform=ax.transAxes,
    va="top",
    fontsize=9,
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.3},
)
ax.text(
    0.98,
    0.98,
    "Innovation\nonly",
    transform=ax.transAxes,
    va="top",
    ha="right",
    fontsize=9,
    bbox={"boxstyle": "round", "facecolor": "lightgreen", "alpha": 0.3},
)
ax.text(
    0.02,
    0.02,
    "Catch-up\nonly",
    transform=ax.transAxes,
    fontsize=9,
    bbox={"boxstyle": "round", "facecolor": "lightblue", "alpha": 0.3},
)
ax.text(
    0.98,
    0.02,
    "Decline",
    transform=ax.transAxes,
    ha="right",
    fontsize=9,
    bbox={"boxstyle": "round", "facecolor": "lightcoral", "alpha": 0.3},
)

ax.set_xlabel("Technical Change ($\\Delta TC$)", fontsize=13)
ax.set_ylabel("Efficiency Change ($\\Delta TE$)", fontsize=13)
ax.set_title("Electricity Sector: TFP Growth Sources by Fuel Type", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "ex2_elec_tc_vs_te.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Step 5: Comparison with manufacturing
# First, re-estimate manufacturing model
mfg_data = load_dataset("manufacturing_panel", category="frontier")
mfg_model = FourComponentSFA(
    data=mfg_data,
    depvar="log_output",
    exog=["log_labor", "log_capital", "log_materials"],
    entity="firm_id",
    time="year",
    frontier_type="production",
)
mfg_result = mfg_model.fit()

mfg_years = sorted(mfg_data["year"].unique())
mfg_tfp = mfg_result.tfp_decomposition(periods=(mfg_years[0], mfg_years[-1]))
mfg_agg = mfg_tfp.aggregate_decomposition()

# Comparison table
comparison = pd.DataFrame(
    {
        "Manufacturing": [
            mfg_agg["mean_delta_tfp"],
            mfg_agg["mean_delta_tc"],
            mfg_agg["mean_delta_te"],
            mfg_agg["mean_delta_se"],
        ],
        "Electricity": [
            elec_agg["mean_delta_tfp"],
            elec_agg["mean_delta_tc"],
            elec_agg["mean_delta_te"],
            elec_agg["mean_delta_se"],
        ],
    },
    index=["TFP Growth", "Technical Change", "Efficiency Change", "Scale Effect"],
)

print("=" * 60)
print("SECTOR COMPARISON: MANUFACTURING vs ELECTRICITY")
print("=" * 60)
display(comparison.round(4))

print("\nKey differences:")
if mfg_agg["mean_delta_tfp"] > elec_agg["mean_delta_tfp"]:
    print("  - Manufacturing has higher TFP growth")
else:
    print("  - Electricity has higher TFP growth")
print(
    f"  - Manufacturing TC: {mfg_agg['pct_from_tc']:.1f}% vs Electricity TC: {elec_agg['pct_from_tc']:.1f}%"
)
print(
    f"  - Manufacturing TE: {mfg_agg['pct_from_te']:.1f}% vs Electricity TE: {elec_agg['pct_from_te']:.1f}%"
)

---

## Exercise 3: Scale Efficiency Analysis (Hard)

Using the manufacturing data and results:
1. Calculate RTS from the four-component model coefficients
2. Compute the scale efficiency change for each firm
3. Create a histogram of the scale effect
4. Determine whether firms with positive scale effects are growing or shrinking
5. Analyze the relationship between firm size and scale effects

In [ ]:
# Step 1: RTS from four-component model
beta_mfg = mfg_result.beta
input_elasticities = beta_mfg[1:]  # Exclude constant
rts = input_elasticities.sum()

print("=" * 60)
print("RETURNS TO SCALE ANALYSIS")
print("=" * 60)
input_names = ["log_labor", "log_capital", "log_materials"]
for name, elast in zip(input_names, input_elasticities):
    print(f"  {name:<20} {elast:>10.4f}")
print(f"  {'=' * 30}")
print(f"  {'RTS':<20} {rts:>10.4f}")
print(f"\n  RTS - 1 = {rts - 1:.4f}")
if rts > 1:
    print("  -> Increasing returns to scale: expansion benefits firms")
elif rts < 1:
    print("  -> Decreasing returns to scale: firms may be over-expanded")
else:
    print("  -> Constant returns to scale: optimal scale")

In [ ]:
# Step 2: Scale effect distribution
mfg_decomp = mfg_tfp.decompose()

print("=" * 60)
print("SCALE EFFECT: DESCRIPTIVE STATISTICS")
print("=" * 60)
print("\n  Scale effect (ΔSE):")
print(f"    Mean:   {mfg_decomp['delta_se'].mean():.4f}")
print(f"    Std:    {mfg_decomp['delta_se'].std():.4f}")
print(f"    Min:    {mfg_decomp['delta_se'].min():.4f}")
print(f"    Max:    {mfg_decomp['delta_se'].max():.4f}")
print(f"    Median: {mfg_decomp['delta_se'].median():.4f}")

pct_positive = 100 * (mfg_decomp["delta_se"] > 0).mean()
print(f"\n  Firms with positive scale effect: {pct_positive:.1f}%")
print(f"  Firms with negative scale effect: {100 - pct_positive:.1f}%")

# Relative magnitude
print("\n  Magnitude comparison (mean absolute values):")
print(f"    |ΔTC|: {mfg_decomp['delta_tc'].abs().mean():.4f}")
print(f"    |ΔTE|: {mfg_decomp['delta_te'].abs().mean():.4f}")
print(f"    |ΔSE|: {mfg_decomp['delta_se'].abs().mean():.4f}")

In [ ]:
# Step 3: Histogram of scale effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(
    mfg_decomp["delta_se"],
    bins=30,
    density=True,
    alpha=0.7,
    color="#f39c12",
    edgecolor="black",
    linewidth=0.5,
)
kde = stats.gaussian_kde(mfg_decomp["delta_se"])
x_range = np.linspace(mfg_decomp["delta_se"].min(), mfg_decomp["delta_se"].max(), 200)
axes[0].plot(x_range, kde(x_range), "k-", linewidth=2)
axes[0].axvline(0, color="red", linestyle="--", linewidth=2, label="Zero (CRS)")
axes[0].axvline(
    mfg_decomp["delta_se"].mean(),
    color="green",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {mfg_decomp['delta_se'].mean():.4f}",
)
axes[0].set_xlabel("Scale Effect ($\\Delta SE$)", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].set_title("Distribution of Scale Effects", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)

# Comparison of all three components
components_data = [
    mfg_decomp["delta_tc"].values,
    mfg_decomp["delta_te"].values,
    mfg_decomp["delta_se"].values,
]
labels_comp = ["$\\Delta TC$", "$\\Delta TE$", "$\\Delta SE$"]
bp = axes[1].boxplot(components_data, labels=labels_comp, patch_artist=True)
colors_box = ["#3498db", "#2ecc71", "#f39c12"]
for patch, color in zip(bp["boxes"], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].axhline(0, color="red", linestyle="--", linewidth=1, alpha=0.5)
axes[1].set_ylabel("Growth Rate", fontsize=12)
axes[1].set_title("Comparison of TFP Components", fontsize=13, fontweight="bold")

plt.suptitle("Scale Efficiency Analysis", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ex3_scale_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Step 4: Are firms with positive scale effects expanding?
# Compute input growth for each firm
data_first = mfg_data[mfg_data["year"] == mfg_years[0]].set_index("firm_id")
data_last = mfg_data[mfg_data["year"] == mfg_years[-1]].set_index("firm_id")

common = data_first.index.intersection(data_last.index)
input_growth = pd.DataFrame(
    {
        "firm_id": common,
        "delta_labor": data_last.loc[common, "log_labor"].values
        - data_first.loc[common, "log_labor"].values,
        "delta_capital": data_last.loc[common, "log_capital"].values
        - data_first.loc[common, "log_capital"].values,
        "delta_materials": data_last.loc[common, "log_materials"].values
        - data_first.loc[common, "log_materials"].values,
    }
)
input_growth["total_input_growth"] = (
    input_growth["delta_labor"] + input_growth["delta_capital"] + input_growth["delta_materials"]
) / 3  # Simple average

# Merge with scale effect
scale_analysis = mfg_decomp[["entity", "delta_se"]].merge(
    input_growth, left_on="entity", right_on="firm_id", how="inner"
)

# Correlation
corr = scale_analysis["delta_se"].corr(scale_analysis["total_input_growth"])
print(f"Correlation between scale effect and input growth: {corr:.4f}")

# Scatter
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    scale_analysis["total_input_growth"],
    scale_analysis["delta_se"],
    alpha=0.5,
    s=40,
    color="steelblue",
    edgecolors="navy",
    linewidth=0.3,
)
ax.axhline(0, color="black", linewidth=0.8, alpha=0.3)
ax.axvline(0, color="black", linewidth=0.8, alpha=0.3)

# Regression line
z = np.polyfit(scale_analysis["total_input_growth"], scale_analysis["delta_se"], 1)
p = np.poly1d(z)
x_line = np.linspace(
    scale_analysis["total_input_growth"].min(), scale_analysis["total_input_growth"].max(), 100
)
ax.plot(x_line, p(x_line), "r-", linewidth=2, label=f"Slope: {z[0]:.4f}")

ax.set_xlabel("Average Input Growth", fontsize=12)
ax.set_ylabel("Scale Effect ($\\Delta SE$)", fontsize=12)
ax.set_title(f"Scale Effect vs Input Growth (corr = {corr:.3f})", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "ex3_scale_vs_growth.png", dpi=300, bbox_inches="tight")
plt.show()

# Interpretation
expanding_pos_se = scale_analysis[
    (scale_analysis["total_input_growth"] > 0) & (scale_analysis["delta_se"] > 0)
]
expanding_neg_se = scale_analysis[
    (scale_analysis["total_input_growth"] > 0) & (scale_analysis["delta_se"] < 0)
]
contracting_pos_se = scale_analysis[
    (scale_analysis["total_input_growth"] < 0) & (scale_analysis["delta_se"] > 0)
]
contracting_neg_se = scale_analysis[
    (scale_analysis["total_input_growth"] < 0) & (scale_analysis["delta_se"] < 0)
]

print("\nFirm classification:")
print(f"  Expanding with positive SE:   {len(expanding_pos_se):>3} firms (benefiting from scale)")
print(f"  Expanding with negative SE:   {len(expanding_neg_se):>3} firms (over-expanded)")
print(f"  Contracting with positive SE: {len(contracting_pos_se):>3} firms (right-sizing)")
print(f"  Contracting with negative SE: {len(contracting_neg_se):>3} firms (shrinking too much)")

In [ ]:
# Step 5: Optimal firm size analysis
# Under Cobb-Douglas with RTS != 1, the scale effect is (RTS - 1) * delta_inputs
# Scale effect = 0 when delta_inputs = 0 (no change) or RTS = 1 (CRS)
# Since our model is Cobb-Douglas, RTS is constant across firms
# The "optimal" insight is about direction: should firms expand or contract?

print("=" * 60)
print("OPTIMAL SCALE ANALYSIS")
print("=" * 60)

print(f"\n  Current RTS: {rts:.4f}")

if rts > 1:
    print("  Regime: Increasing Returns to Scale")
    print("  -> All firms benefit from expansion")
    print(f"  -> Scale effect = ({rts:.4f} - 1) * weighted_input_change = {rts - 1:.4f} * Δinputs")
    print(f"  -> A 10% expansion in all inputs yields {10 * (rts - 1):.2f}% scale bonus")
elif rts < 1:
    print("  Regime: Decreasing Returns to Scale")
    print("  -> Expansion has negative scale effects")
    print(f"  -> A 10% expansion in all inputs yields {10 * (rts - 1):.2f}% scale penalty")
    print("  -> Firms would benefit from right-sizing or specialization")

# Analyze firm size distribution and relationship with efficiency
# Use initial period output as proxy for firm size
firm_size = data_first["log_output"].to_frame().reset_index()
firm_size.columns = ["firm_id", "initial_size"]

# Merge with scale effect and efficiency
size_eff = scale_analysis.merge(firm_size, on="firm_id", how="inner")

# Size quartiles
size_eff["size_quartile"] = pd.qcut(
    size_eff["initial_size"], 4, labels=["Small", "Medium-Small", "Medium-Large", "Large"]
)

print("\nScale effect by firm size:")
size_se = size_eff.groupby("size_quartile")["delta_se"].agg(["mean", "std", "count"])
display(size_se.round(4))

print("\nInterpretation:")
print(f"  Under {'IRS' if rts > 1 else 'DRS'}, the scale effect is proportional to input growth.")
print(
    f"  Firms that grow faster have {'larger scale gains' if rts > 1 else 'larger scale losses'}."
)
print("  The 'optimal' firm size under Cobb-Douglas with constant RTS is not well-defined;")
print("  the translog specification would be needed to identify a unique optimal scale.")

---

*End of solutions for Tutorial 11C.*